# Larger LLMs for schema discovery, smaller LLMs for application

We build one **Elastic Workflow** that asks: *what does this report reveal about the pilot who wrote it?* Since [NASA ASRS](https://asrs.arc.nasa.gov/) reports are anonymous, "about the pilot" means behavioral traits visible in the narration (decision-making style, situational awareness, accountability), not identity.
A large reasoning model reads a stratified sample of reports and proposes a categorical schema. A human approves it from the Kibana UI. A small [Mistral](https://mistral.ai/) model applies the schema across the full corpus: we pay for deep reasoning once, and the cheap model handles the thousands of repetitive classifications.

## Prerequisites

- Elastic Stack 9.4+ or Elastic Cloud Serverless. Workflows has been GA since 9.4.
- [Agent Builder](https://www.elastic.co/docs/solutions/search/elastic-agent-builder) enabled in your deployment.
- A [Kibana GenAI connector](https://www.elastic.co/docs/reference/kibana/connectors-kibana/ai-connector) pointing at Claude Sonnet (or an equivalent reasoning model). This is the planner.
- A [Mistral API key](https://console.mistral.ai/api-keys). We use it to register an Elasticsearch inference endpoint.
- Python 3.10+. Dependencies are pinned in [`requirements.txt`](requirements.txt) and installed by the `%pip install` cell below.

## Setup

In [1]:
%pip install -qr requirements.txt

Reason for being yanked: Yanked due to conflicts with CVE-2024-35195 mitigation
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.0 requires requests>=2.32.2, but you have requests 2.32.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Create the corpus index

The index mappings match the ASRS columns we will use: `flight_phase` and `anomaly` for stratification, `narrative` and `synopsis` as document text.

In [3]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected to Elasticsearch: {es.info()['version']['number']}")

Connected to Elasticsearch: 9.4.2


In [4]:
INDEX = "incident_reports"

mappings = {
    "properties": {
        "acn": {"type": "keyword"},
        "flight_phase": {"type": "keyword"},
        "anomaly": {"type": "keyword"},
        "synopsis": {"type": "text"},
        "narrative": {"type": "text"},
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)

es.indices.create(index=INDEX, mappings=mappings)
print(f"Created index: {INDEX}")

Created index: incident_reports


## Load ASRS reports

The dataset (`asrs.csv`) was extracted from the [NASA ASRS Database Online](https://asrs.arc.nasa.gov/search/database.html) using the public search interface and exported as CSV. It is bundled next to this notebook in the repository.

In [5]:
import pandas as pd

# ASRS CSV exports have a category row above the actual column names.
# Skip the category row; pandas drops the blank line that follows automatically.
df = pd.read_csv("asrs.csv", skiprows=[0], skip_blank_lines=True, low_memory=False)

# Some column names repeat (e.g. Aircraft 1/Aircraft 2 both have "Flight Phase");
# pandas renames duplicates with a ".1" suffix. We just use the first occurrence.
df = df.dropna(subset=["ACN", "Narrative"])

# Optional columns (Flight Phase, Anomaly, Synopsis) can be empty; pandas reads
# those as NaN, which is not valid JSON, so replace them with None.
df = df.where(pd.notna(df), None)


def to_doc(row):
    return {
        "_index": INDEX,
        "_id": str(row["ACN"]),
        "_source": {
            "acn": str(row["ACN"]),
            "flight_phase": row.get("Flight Phase"),
            "anomaly": row.get("Anomaly"),
            "synopsis": row.get("Synopsis"),
            "narrative": row.get("Narrative"),
        },
    }


# raise_on_error=False: a handful of rows in the public ASRS export are
# malformed. For this demo we skip those documents and report the count below
# instead of aborting the whole ingest; drop the flag if you prefer to fail
# fast and inspect the offending rows.
ok, errors = helpers.bulk(
    es,
    (to_doc(r) for _, r in df.iterrows()),
    raise_on_error=False,
)
print(f"indexed {ok}, errors {len(errors)}")

indexed 4287, errors 0


## Register the Mistral inference endpoint

The classification step calls this endpoint once per document via `ai.agent`.

In [7]:
INFERENCE_ID = "mistral-small-extractor"

# Delete and re-create so the rate_limit setting picks up changes between runs.
# 6 RPM = 1 every 10s; conservative for Mistral free tier to avoid 429s.
es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

es.inference.put(
    task_type="chat_completion",
    inference_id=INFERENCE_ID,
    inference_config={
        "service": "mistral",
        "service_settings": {
            "api_key": MISTRAL_API_KEY,
            "model": "mistral-small-latest",
            "rate_limit": {"requests_per_minute": 6},
        },
    },
)
print(f"Inference endpoint ready: {INFERENCE_ID}")

BadRequestError: BadRequestError(400, 'status_exception', 'unified_chat_completion_exception: Received an authentication error status code for request from inference entity id [mistral-small-extractor] status [401]. Error message: [{"detail":"Unauthorized"}\n]', unified_chat_completion_exception: Received an authentication error status code for request from inference entity id [mistral-small-extractor] status [401]. Error message: [{"detail":"Unauthorized"}
])

## Auxiliary indices

`schemas` stores one document per discovery run; `extractions` stores one document per source report per schema version. Keeping them separate from the corpus gives an audit trail of every approved schema and lets us re-run classification with a revised schema without touching the source reports.

In [ ]:
for aux in ["schemas", "extractions"]:
    if not es.indices.exists(index=aux):
        es.indices.create(index=aux)
        print(f"Created index: {aux}")

Created index: extractions


## Create the workflow via the Workflows API

The workflow is defined in [`workflow.yaml`](workflow.yaml) and registered via `POST /api/workflows/workflow`. This is a Kibana API, not an Elasticsearch one, so we use plain `requests` instead of the Elasticsearch client. The steps, in order:

1. **Stratified sample**: two aggregations pull a small sample, by flight phase and by anomaly type.
2. **Discover**: the large model proposes a schema from the sample.
3. **Human gate**: execution pauses until the schema is approved in Kibana.
4. **Store schema**: the approved schema is saved to `schemas`.
5. **Fetch corpus**: the reports to classify are retrieved.
6. **Classify (foreach)**: the Mistral model applies the schema to each report and writes to `extractions`.

Both models are provider-hosted, not in your cluster: the planner behind the Kibana GenAI connector ([EIS](https://www.elastic.co/docs/explore-analyze/elastic-inference/eis), Bedrock, OpenAI, or Gemini), the executor behind the Elasticsearch inference endpoint calling Mistral's API.

In [ ]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Discover the GenAI connector ID (large reasoning model) from Kibana.
# Looks for any Bedrock / OpenAI / Gemini / generic GenAI connector.
GENAI_TYPES = {".bedrock", ".gen-ai", ".gemini", ".inference"}

response = requests.get(f"{KIBANA_URL}/api/actions/connectors", headers=headers)
response.raise_for_status()

genai_connectors = [c for c in response.json() if c["connector_type_id"] in GENAI_TYPES]
if not genai_connectors:
    raise RuntimeError(
        "No GenAI connectors found. Create one in Kibana > Stack Management > Connectors."
    )

KIBANA_CONNECTOR_ID = genai_connectors[0]["id"]
print(f"Using connector: {genai_connectors[0]['name']} ({KIBANA_CONNECTOR_ID})")

Using connector: Anthropic Claude Haiku 4.5 (Anthropic-Claude-Haiku-4-5)


In [ ]:
from pathlib import Path

WORKFLOW_NAME = "Pilot Schema Discovery and Application"

# The full workflow definition lives in workflow.yaml next to this notebook.
# Only the environment-specific values are substituted here.
workflow_yaml = (
    Path("workflow.yaml")
    .read_text()
    .replace("__CORPUS_INDEX__", INDEX)
    .replace("__CONNECTOR_ID__", KIBANA_CONNECTOR_ID)
    .replace("__INFERENCE_ID__", INFERENCE_ID)
)

In [ ]:
WORKFLOW_ID = "pilot-schema-discovery"

# Delete any existing workflow with the same id (allows re-runs)
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)

# Create the workflow from the YAML definition, keeping our custom id.
response = requests.post(
    f"{KIBANA_URL}/api/workflows/workflow",
    headers=headers,
    json={"id": WORKFLOW_ID, "yaml": workflow_yaml},
)
response.raise_for_status()

print(f"Workflow ready: {WORKFLOW_ID} ({WORKFLOW_NAME})")
print(f"Open in Kibana: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}")

## Trigger the workflow

`POST /api/workflows/workflow/{id}/run` starts an execution and returns a `workflowExecutionId`. The run will pause at the `human_gate` step until the schema is approved from the Kibana UI (Workflows > Executions > the running one). After approval, the classification step fans out and writes to the `extractions` index.

In [ ]:
response = requests.post(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}/run",
    headers=headers,
    json={
        "inputs": {"goal": "What does this report reveal about the pilot who wrote it?"}
    },
)
if response.status_code == 404:
    raise RuntimeError(
        "Workflow not found. Run the registration cell above first, and make "
        "sure the deployment is on 9.4+ (the Workflows API is GA since 9.4)."
    )
response.raise_for_status()

execution_id = response.json()["workflowExecutionId"]
print(f"Execution started: {execution_id}")
print(
    f"Approve the schema at: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}"
    f"?tab=executions&executionId={execution_id}"
)

## Print the discover step output

`waitForInput` cannot be pre-populated from a previous step today (the API exposes only `message` and `schema`). To make the human gate practical, we poll the execution until the `discover` step completes and pretty-print its output. Copy that JSON, edit it if needed, and paste it into the `approved_fields` field of the form.

In [ ]:
import time

# execution_id is defined by the trigger cell above; this guard gives a
# clearer message than the NameError you would get otherwise.
if "execution_id" not in globals():
    raise RuntimeError("execution_id is not defined. Run the trigger cell above first.")


def step_output(execution_id, step_id, timeout=120, interval=3):
    """Poll an execution until the given step completes, then return its output."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        exec_response = requests.get(
            f"{KIBANA_URL}/api/workflows/executions/{execution_id}", headers=headers
        )
        exec_response.raise_for_status()
        step = next(
            (
                s
                for s in exec_response.json().get("stepExecutions", [])
                if s.get("stepId") == step_id
                and s.get("status") in ("completed", "failed")
            ),
            None,
        )
        if step:
            step_response = requests.get(
                f"{KIBANA_URL}/api/workflows/executions/{execution_id}/step/{step['id']}",
                headers=headers,
            )
            step_response.raise_for_status()
            detail = step_response.json()
            if step.get("status") == "failed":
                raise RuntimeError(
                    f"step '{step_id}' failed: {detail.get('error') or detail}"
                )
            return detail.get("output")
        time.sleep(interval)
    raise TimeoutError(f"step '{step_id}' did not complete in time")


discover = step_output(execution_id, "discover")

# The proposed schema lives under `content` in the step output. Validate the
# shape before using it so a connector problem surfaces as a readable error
# instead of a TypeError.
content = (discover or {}).get("content") or {}
if not content.get("fields"):
    raise RuntimeError(
        f"Unexpected discover output, check the execution in Kibana: {discover!r}"
    )

# Strip `why_useful` (not part of the human_gate form) and wrap in the shape
# expected by the waitForInput form so this is paste-ready.
approved_fields = [
    {
        "name": field["name"],
        "definition": field["definition"],
        "values": [
            {
                "value": v["value"],
                "definition": v["definition"],
            }
            for v in field["values"]
        ],
    }
    for field in content["fields"]
]

print(json.dumps({"approved_fields": approved_fields, "notes": ""}, indent=2))

## Cleanup

In [ ]:
for idx in [INDEX, "schemas", "extractions"]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)

es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

In [ ]:
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)
print(f"Deleted workflow: {WORKFLOW_ID}")